# Task 2: Quantitative Analysis using TA-Lib

## Objective
Load historical stock price data, compute financial technical indicators (Moving Averages, RSI, MACD), and visualize the results to understand market behavior.

---

In [ ]:
import pandas as pd
import numpy as np
import talib
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline
print("Libraries (TA-Lib, Pandas, Matplotlib) imported successfully.")

## 1. Prepare Your Data

### 1.a & 1.b Load and Type Data
We load the Apple Inc. (AAPL) stock dataset and ensure correct typing for Open, High, Low, Close, and Volume.

In [ ]:
stock_file = "../Data/yfinance_data/Data/AAPL.csv"
df = pd.read_csv(stock_file)

df['Date'] = pd.to_datetime(df['Date'])
df.sort_values('Date', inplace=True)
df.set_index('Date', inplace=True)

print("Data Preparation Complete. Rows:", len(df))
df.head()

### 1.c Handle Missing Values
We check for and address any missing data points using forward-filling.

In [ ]:
if df.isnull().sum().any():
    print("Handling missing values...")
    df = df.ffill()

print("Missing values handled.")

## 2. Compute Technical Indicators with TA-Lib

### 2.1 Moving Averages (MA): SMA and EMA

In [ ]:
close_prices = df['Close'].values

# Calculate 20-day Simple Moving Average (SMA)
df['SMA_20'] = talib.SMA(close_prices, timeperiod=20)

# Calculate 20-day Exponential Moving Average (EMA)
df['EMA_20'] = talib.EMA(close_prices, timeperiod=20)

print("SMA and EMA calculated.")

### 2.2 Relative Strength Index (RSI)
Identifying overbought (>70) and oversold (<30) conditions.

In [ ]:
df['RSI'] = talib.RSI(close_prices, timeperiod=14)
print("RSI calculated.")

### 2.c MACD (Moving Average Convergence Divergence)
Detecting momentum shifts and potential trend reversals.

In [ ]:
macd, signal, hist = talib.MACD(close_prices, fastperiod=12, slowperiod=26, signalperiod=9)
df['MACD'] = macd
df['MACD_Signal'] = signal
df['MACD_Hist'] = hist

print("MACD calculated.")

## 3. Apply Financial Metrics

We compute additional metrics such as daily returns and annualized volatility.

In [ ]:
df['Daily_Return'] = df['Close'].pct_change()
ann_vol = df['Daily_Return'].std() * np.sqrt(252)

print(f"Annualized Volatility: {ann_vol:.4f}")

## 4. Visualize the Data

### 4.a, 4.b, 4.c Integrated Dashboard
Visualizing price action overlaid with moving averages, alongside separate panels for RSI and MACD.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 15), sharex=True)

# 4.a Plot closing prices overlaid with moving averages
ax1.plot(df.index, df['Close'], label='Close Price', color='black', alpha=0.3)
ax1.plot(df.index, df['SMA_20'], label='SMA 20 (Slow)', color='blue')
ax1.plot(df.index, df['EMA_20'], label='EMA 20 (Fast)', color='orange')
ax1.set_title('AAPL Stock Price & Trends', fontsize=16)
ax1.legend()

# 4.b Plot RSI
ax2.plot(df.index, df['RSI'], color='purple')
ax2.axhline(70, color='red', linestyle='--', label='Overbought')
ax2.axhline(30, color='green', linestyle='--', label='Oversold')
ax2.set_title('Relative Strength Index (RSI)', fontsize=14)
ax2.legend(loc='upper left')

# 4.c Plot MACD
ax3.plot(df.index, df['MACD'], label='MACD', color='blue')
ax3.plot(df.index, df['MACD_Signal'], label='Signal', color='orange')
ax3.bar(df.index, df['MACD_Hist'], label='Momentum', color='gray', alpha=0.2)
ax3.set_title('MACD Momentum', fontsize=14)
ax3.legend(loc='upper left')

plt.tight_layout()
plt.show()

### Insight - Technical Analysis Summary:
- **Trend Identification:** The **EMA 20 (Orange)** acts as a support line during the steady uptrend from 2018 onwards. Because it reacts faster than the **SMA 20**, it successfully captured the rapid price recovery seen in late 2020 and 2023.
- **Momentum Overextension:** The **RSI** shows multiple spikes above the **70 (Overbought)** line, which often preceded short-term price consolidations. Conversely, dips below **30 (Oversold)** marked potential high-probability entry points during the 2022 market volatility.
- **Volatility & Momentum:** The **MACD** histogram shows significant momentum expansion starting in 2020, reflecting the increased trading volume and volatility in the tech sector during that period.

## 5. Summary of Preliminary Findings

### Data Preparation
- **Cleaning:** Successfully handled datetime conversion and missing data. We addressed the **Timezone Standardization** challenge from Task 1, ensuring data integrity for future sentiment correlation.

### Technical Analysis
- **Moving Averages:** The **EMA 20 (Orange)** correctly followed the price tighter and reacted faster to trend changes compared to the **SMA 20 (Blue)**.
- **Indicators:** RSI successfully flagged overbought periods, and the MACD histogram clearly visualized momentum shifts.

### Future Strategy
- We plan to correlate news sentiment spikes with these technical momentum signals in the final phase.